In [41]:
import pandas as pd
import numpy as np

In [58]:
team_stats = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/team_stats.csv')

In [76]:
team_stats = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/team_stats.csv')

team_stats = team_stats.drop(columns='Unnamed: 0')
team_stats = team_stats.rename(columns={'Cmp-Att-Yd-TD-INT':'ManyStats'})
team_stats[['Completion', 'Attempts', 'Yards', 'Touchdowns', 'Interceptions']] = team_stats.ManyStats.str.split('-', expand=True)
team_stats = team_stats.drop(columns='ManyStats')

team_stats.columns = team_stats.columns.str.replace(' ', '_')
team_stats.columns = team_stats.columns.str.replace('.', '_')
team_stats.columns = team_stats.columns.str.replace('-', '_')

team_stats[['Success4thDown', 'Failed4thDown']] = team_stats.Fourth_Down_Conv_.str.split('-', expand=True)
team_stats[['Fumbles', 'FumblesLost']] = team_stats.Fumbles_Lost.str.split('-', expand=True)
team_stats[['Penalties', 'PenaltyYards']] = team_stats.Penalties_Yards.str.split('-', expand=True)
#team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats[['Sacked', 'SackYards']] = team_stats.Sacked_Yards.str.split('-', expand=True)
team_stats[['Success3rdDown', 'Failed3rdDown']] = team_stats.Third_Down_Conv_.str.split('-', expand=True)
team_stats[['TimeMinutes', 'TimeSeconds']] = team_stats.Time_of_Possession.str.split(':', expand=True)

team_stats = team_stats.drop(columns={'Fourth_Down_Conv_', 'Fumbles_Lost', 'Penalties_Yards', 'Sacked_Yards', 'Third_Down_Conv_', 'Time_of_Possession'})

team_stats['Rush_Yds_TDs'].str.count('-').value_counts()

team_stats.loc[2083]

team_stats.at[2083,'Rush_Yds_TDs'] = '8-neg1-0'

team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats = team_stats.drop(columns='Rush_Yds_TDs')

team_stats.at[2083, "RushYards"] = -1

skip_cols = ['team_name', 'date']
cols_to_change = [col for col in team_stats.columns if col not in skip_cols]
team_stats[cols_to_change] = team_stats[cols_to_change].astype(int)

team_stats['CompletionPct'] = round(100 * (team_stats['Completion'] / team_stats['Attempts']), 1)
team_stats['PctYardsPass'] = round(100 * (team_stats['Yards'] / team_stats['Total_Yards']), 1)
team_stats['PctYardsRush'] = round(100 * (team_stats['RushYards'] / team_stats['Total_Yards']), 1)
team_stats['SuccessRate4thDown'] = round(
    100 * (team_stats['Success4thDown'] / (team_stats['Success4thDown'] + team_stats['Failed4thDown'])), 1)
team_stats['SuccessRate3rdDown'] = round(
    100 * (team_stats['Success3rdDown'] / (team_stats['Success3rdDown'] + team_stats['Failed3rdDown'])), 1)
team_stats['PossessionTime'] = (60 * team_stats['TimeMinutes']) + team_stats['TimeSeconds']

team_stats[['DayofWeek', 'Month', 'DayofMonth', 'Year']] = team_stats.date.str.split(' ', expand=True)
team_stats['DayofMonth'] = team_stats['DayofMonth'].str.rstrip(',')

game_months = [
    (team_stats['Month'] == 'Sep'),
    (team_stats['Month'] == 'Oct'),
    (team_stats['Month'] == 'Nov'),
    (team_stats['Month'] == 'Dec'),
    (team_stats['Month'] == 'Jan'),
    (team_stats['Month'] == 'Feb')
]
month_choices = [9, 10, 11, 12, 1, 2]

team_stats['Month'] = np.select(game_months, month_choices, default=False)
team_stats['DayofMonth'] = team_stats['DayofMonth'].astype(int)
team_stats['Year'] = team_stats['Year'].astype(int)

date_mapping = {
    'Year':'year',
    'DayofMonth':'day',
    'Month':'month'}

team_stats = team_stats.rename(mapper=date_mapping, axis=1)
team_stats['gamedate'] = pd.to_datetime(team_stats[['year', 'month', 'day']]).dt.strftime('%d-%m-%Y')

team_stats = team_stats.drop(columns={'date','Completion','Attempts','Success4thDown','Failed4thDown','Success3rdDown','Failed3rdDown'})

In [77]:
team_stats

,First_Downs,Net_Pass_Yards,Total_Yards,Turnovers,team_name,Yards,Touchdowns,Interceptions,Fumbles,FumblesLost,...,PctYardsPass,PctYardsRush,SuccessRate4thDown,SuccessRate3rdDown,PossessionTime,DayofWeek,month,day,year,gamedate
0,19,283,389,3,GNB,291,1,3,2,0,...,74.8,27.2,20.0,34.8,2082,Sunday,11,6,2022,06-11-2022
1,19,137,254,1,DET,137,2,1,0,0,...,53.9,46.1,0.0,35.3,1518,Sunday,11,6,2022,06-11-2022
2,20,127,338,2,PHI,154,1,1,2,1,...,45.6,62.4,50.0,15.8,2178,Sunday,12,22,2024,22-12-2024
3,23,255,368,5,WAS,258,5,2,3,3,...,70.1,30.7,40.0,35.0,1422,Sunday,12,22,2024,22-12-2024
4,23,204,311,0,DAL,204,2,0,0,0,...,65.6,34.4,NaN,30.4,1881,Sunday,11,19,2023,19-11-2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,16,119,193,1,PIT,148,1,0,2,1,...,76.7,38.3,0.0,25.0,1320,Saturday,1,4,2025,04-01-2025
2274,19,242,339,1,DAL,253,2,0,1,1,...,74.6,28.6,50.0,25.0,1862,Sunday,12,24,2023,24-12-2023
2275,22,284,375,0,MIA,293,1,0,1,0,...,78.1,24.3,0.0,31.6,1738,Sunday,12,24,2023,24-12-2023
2276,18,171,264,1,WAS,191,1,1,1,0,...,72.3,35.2,33.3,20.0,1616,Thursday,11,14,2024,14-11-2024
